# Database Setup

Sets up the PostgreSQL database and prepares the datasets required for the provenance analysis.

In [10]:
import pandas as pd
import os

file_names = [
    "df_control.csv",
    "df_1_20.csv",
    "df_1_7_9.csv",
    "df_5_20.csv",
    "df_5_7_9.csv"
]

dfs = []
for name in file_names:
    if os.path.exists(name):
        df = pd.read_csv(name)
        dfs.append(df)
        print(f"{name} ({df.shape[0]} rows, {df.shape[1]} columns) has been loaded")
    else:
        print(f"{name} not found – skip")

if dfs:
    all_df = pd.concat(dfs, ignore_index=True)
    print(f"\n✅ All DataFrames merged: {all_df.shape[0]} rows, {all_df.shape[1]} columns")

    all_df["provid"] = ["ti_" + str(i) for i in range(1, len(all_df) + 1)]

    out_file = "all.csv"
    all_df.to_csv(out_file, index=False)
    print(f"\n✅ Results saved with a new column in '{out_file}'")
else:
    print("\n❌ No data was read – no file was created.")

df_control.csv (240 rows, 62 columns) has been loaded
df_1_20.csv (240 rows, 62 columns) has been loaded
df_1_7_9.csv (240 rows, 62 columns) has been loaded
df_5_20.csv (240 rows, 62 columns) has been loaded
df_5_7_9.csv (240 rows, 62 columns) has been loaded

✅ All DataFrames merged: 1200 rows, 62 columns

✅ Results saved with a new column in 'all.csv'


In [9]:
import psycopg2
import pandas as pd
from io import StringIO

conn = psycopg2.connect(
    dbname="DBplyr",
    user="admin",
    password="...",
    host="localhost",
    port="5432"
)
cursor = conn.cursor()
print("✅ Connection to the database established")

type_map = {
    'int64':        'BIGINT',
    'float64':      'DOUBLE PRECISION',
    'object':       'TEXT',
    'bool':         'BOOLEAN',
    'datetime64[ns]': 'TIMESTAMP',
    'datetime64[ns, tz]': 'TIMESTAMP WITH TIME ZONE',
}

table_name = "ti"

col_defs = []
for col in all_df.columns:
    pg_type = type_map.get(str(all_df[col].dtype), 'TEXT')
    col_defs.append(f'"{col.lower()}" {pg_type}')

create_sql = f"""
CREATE TABLE IF NOT EXISTS {table_name} (
    {', '.join(col_defs)}
);
"""
cursor.execute(create_sql)
conn.commit()
print(f"✅ Table {table_name} created (if it did not exist)")

csv_buffer = StringIO()
all_df.to_csv(csv_buffer, sep='\t', header=False, index=False, na_rep='\\N')
csv_buffer.seek(0)

copy_sql = f"""
COPY {table_name} FROM STDIN WITH (FORMAT csv, DELIMITER '\t', NULL '\\N')
"""
cursor.copy_expert(copy_sql, csv_buffer)
conn.commit()
print(f"✅ Data written to table {table_name}")

cursor.close()
conn.close()
print("✅ Connection closed")

✅ Verbindung zur Datenbank hergestellt
✅ Tabelle ti angelegt (falls nicht vorhanden)
✅ Daten in Tabelle ti geschrieben
✅ Verbindung geschlossen
